Analyze MFI (Mean Fluorescence Intensities) in cells.


Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from skimage import exposure
from skimage.filters import gaussian, threshold_otsu

from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, binary_dilation, disk
from skimage.segmentation import find_boundaries

from cellpose import models, denoise
from cellpose.utils import fill_holes_and_remove_small_masks

from aicspylibczi import CziFile

model = models.Cellpose(model_type='cyto3')
nuclei_model = models.Cellpose(model_type='nuclei')
model_3d = denoise.CellposeDenoiseModel(model_type="cyto3",
             restore_type="denoise_cyto3", chan2_restore=True)

In [ ]:
MIN_INCLUSION_SIZE = 10
MAX_INCLUSION_SIZE = 10000
COLOR_SCHEME = 'viridis'
RADIUS = 7

# debug parameters
DISPLAY_INCLUSION_GRAPH = False
VERBOSE = False

Define Sub Functions

In [ ]:
def show_image(image, title=None, cmap=COLOR_SCHEME):
    """Display a single image with an optional title and colorbar."""
    plt.figure(figsize=(8, 6))
    plt.imshow(image, cmap=cmap)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.colorbar()
    plt.tight_layout()
    plt.show()

def show_images(images, titles=None, cmap=COLOR_SCHEME):
    """Display a grid of images with optional titles and colorbars."""
    n = len(images)
    cols = min(n, 5)
    rows = -(-n // cols)  # ceiling division
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = np.atleast_2d(axes).reshape(rows, cols)

    for i, img in enumerate(images):
        ax = axes[i // cols, i % cols]
        im = ax.imshow(img, cmap=cmap)
        if titles:
            ax.set_title(titles[i])
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for j in range(i + 1, rows * cols):
        axes[j // cols, j % cols].axis('off')

    plt.tight_layout()
    plt.show()



def normalize_image(image):
    """Normalize image to [0, 1]."""
    min_val, max_val = np.min(image), np.max(image)
    return image if max_val - min_val == 0 else (image - min_val) / (max_val - min_val)

def preprocess_cellpose_image(image):
    """Prepare image for Cellpose: normalize, gamma correct, clip, and blur."""
    image = normalize_image(image)
    image = gaussian(image, sigma=2)
    image = normalize_image(image ** 0.01)  # gamma correction
    clip_threshold = np.percentile(image, 90)
    image = np.clip(image, 0, clip_threshold) / clip_threshold
    return normalize_image(gaussian(image, sigma=2))

def segment_cells(green_channel, blue_channel=None):
    """Segment 2D cells. Use green only, or green + blue as channels."""
    if blue_channel is None:
        channel_data = preprocess_cellpose_image(green_channel)
        channels = [0, 0]
    else:
        channel_data = np.stack([green_channel, blue_channel], axis=0)
        channels = [1, 2]
  

    for diameter in range(200, 700, 25):
        masks, _, _, _ = model.eval(
            channel_data,
            diameter=diameter,
            channels=channels,
        )
        labeled = label(masks)
        if np.max(labeled) > 0:
            return labeled

    print("No cells found.")
    return np.zeros_like(green_channel)


def segment_cells_3D(green_channels, blue_channels=None):
    """Segment 3D cells across z-slices."""
    if blue_channels is None:
        # preprocess each slice separately
        preprocessed_slices = np.stack(
            [preprocess_cellpose_image(green_channels[z]) for z in range(green_channels.shape[0])],
            axis=0
        )
        channel_data = preprocessed_slices
        channels = [0, 0]
    else:
        channel_data = np.stack([green_channels, blue_channels], axis=1)
        channels = [1, 2]

    for diameter in range(200, 700, 25):
        masks, _, _, _ = model.eval(
            channel_data,
            diameter=diameter,
            channels=channels,
            channel_axis=1,
            do_3D=False, # use 2D model on each slice and then stitch
            anisotropy=10.5,
            min_size=10000,
            stitch_threshold=0.5, 
        )

        if np.sum(masks) > 0:
            return [label(masks[i]) for i in range(masks.shape[0])]

    print("No cells found.")
    return np.zeros_like(green_channels)

def calculate_circularity_index(mask):
    """Calculate circularity index of a binary mask."""
    if np.sum(mask) == 0:
        return 0
    area = np.sum(mask)
    perimeter = np.sum(find_boundaries(mask, mode='outer', connectivity=2))
    return (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

def extract_inclusions(green_channel, cell_mask, no_inclusions, display_graph=False):
    """Detect inclusions inside a cell mask using intensity distribution."""
    if no_inclusions:
        return np.zeros_like(green_channel, dtype=bool)

    green_channel = normalize_image(green_channel)
    blurred = gaussian(green_channel, sigma=2)
    masked = blurred * cell_mask
    values = normalize_image(masked[masked > 0])

    q1 = np.percentile(values, 25)
    q3 = np.percentile(values, 75)
    hist, bins = np.histogram(values, bins='fd')

    if q3 < 0.4 and len(bins) > 20:
        threshold = max(threshold_otsu(green_channel * cell_mask), 0.4) 
    elif q3 >= 0.7:
        threshold = 1
    else:
        threshold = 0.999

    binary = (green_channel * cell_mask) > threshold
    filtered = remove_small_objects(binary, MIN_INCLUSION_SIZE)
    inclusions = filtered ^ remove_small_objects(filtered, MAX_INCLUSION_SIZE)
    inclusions_labeled = label(inclusions)
    # remove inclusions with circularity index < 0.1
    for region in regionprops(inclusions_labeled):
        
        circularity = calculate_circularity_index(inclusions_labeled == region.label)
        if circularity < 0.3:
            inclusions[inclusions_labeled == region.label] = False
        

    if display_graph:
        plt.hist(values, bins='fd')
        for val, _label, color in zip(
            [np.mean(values), np.median(values), q1, q3, threshold],
            ['Mean', 'Median', 'Q1', 'Q3', 'Threshold'],
            ['red', 'green', 'blue', 'purple', 'orange'],
        ):
            plt.axvline(val, color=color, linestyle='--', label=f'{_label}: {val:.2f}')
        plt.legend()
        plt.title("Intensity Histogram")
        plt.show()

        show_image(inclusions, title="Inclusions")

    return inclusions


def generate_inclusion_image(green_channel, labeled_cells, no_inclusions):
    """Return binary inclusion image from labeled cells."""
    if no_inclusions:
        return np.zeros_like(green_channel, dtype=bool)
    inclusion_map = np.zeros_like(green_channel)

    for region in regionprops(labeled_cells):
        if region.area < 100:
            continue
        mask = labeled_cells == region.label
        inclusion_map += extract_inclusions(green_channel, mask, no_inclusions, display_graph=DISPLAY_INCLUSION_GRAPH)

    return inclusion_map






def calculate_mfi_around_inclusions(inclusions_mask, mitochondria_image, cell_mask,verbose=False):
    """
    inclusion_mask: binary mask of the inclusions IN THE CELL
    mitochondria_image: image of the mitochondria
    cell_mask: binary mask of the cell
    """
    mfi_cell = calculate_mean_intensity(mitochondria_image, cell_mask)
    mfi_inclusions = calculate_mean_intensity(mitochondria_image, inclusions_mask)

    dilated_inclusion_mask = binary_dilation(inclusions_mask, disk(RADIUS))
    dilated_inclusion_mask = dilated_inclusion_mask * cell_mask 
    mfi_inclusions_and_surroundings = calculate_mean_intensity(mitochondria_image, dilated_inclusion_mask)

    surroundings_mask = dilated_inclusion_mask ^ inclusions_mask  # this will remove the inclusion image from the dilated image
    surroundings_mask = surroundings_mask * cell_mask 
    mfi_surroundings = calculate_mean_intensity(mitochondria_image, surroundings_mask)

    
    mask_without_inclusions_and_surroundings = cell_mask ^ dilated_inclusion_mask
    mfi_without_inclusions_and_surroundings = calculate_mean_intensity(mitochondria_image, mask_without_inclusions_and_surroundings)

    mask_without_inclusions = cell_mask ^ inclusions_mask
    mfi_without_inclusions = calculate_mean_intensity(mitochondria_image, mask_without_inclusions)

    if np.sum(inclusions_mask) == 0:
        mfi_without_inclusions = None
        mfi_without_inclusions_and_surroundings = None
   
    if verbose:
        inclusion_image_combined = inclusions_mask.copy().astype(np.uint8)
        border = find_boundaries(dilated_inclusion_mask, mode='outer', connectivity=2)
        inclusion_image_combined[border] = 2
        show_images([inclusion_image_combined, surroundings_mask, mask_without_inclusions, mask_without_inclusions_and_surroundings],
                      titles=["Inclusions and Surroundings","Surroundings", "Without Inclusions", "Without Inclusions and Surroundings"],
                      cmap=COLOR_SCHEME)
       
        print(f"MFI Cell: {mfi_cell}, MFI Inclusions: {mfi_inclusions}, MFI Surroundings: {mfi_surroundings}, MFI Inclusions and Surroundings: {mfi_inclusions_and_surroundings}, MFI Without Inclusions: {mfi_without_inclusions}, MFI Without Inclusions and Surroundings: {mfi_without_inclusions_and_surroundings}")

    return mfi_cell, mfi_inclusions, mfi_surroundings, mfi_inclusions_and_surroundings ,mfi_without_inclusions, mfi_without_inclusions_and_surroundings



def calculate_density_around_inclusions(inclusions_mask, mitochondria_image, cell_mask_remove_nuclei, verbose=False):
    """
    inclusion_mask: binary mask of the inclusions in the cell
    mitochondria_image: image of the mitochondria
    cell_mask_remove_nuclei: binary mask of the cell without nuclei
    """

    inclusions_mask = inclusions_mask * cell_mask_remove_nuclei  
    density_cell =calculate_density(mitochondria_image, cell_mask_remove_nuclei)
    density_inclusions = calculate_density(mitochondria_image, inclusions_mask)

    dilated_inclusion_mask = binary_dilation(inclusions_mask, disk(RADIUS))
    dilated_inclusion_mask = dilated_inclusion_mask * cell_mask_remove_nuclei 
    density_inclusions_and_surroundings = calculate_density(mitochondria_image, dilated_inclusion_mask)


    surroundings_mask = dilated_inclusion_mask ^ inclusions_mask  # this will remove the inclusion image from the dilated image
    surroundings_mask = surroundings_mask * cell_mask_remove_nuclei 
    density_surroundings = calculate_density(mitochondria_image, surroundings_mask)

    mask_without_inclusions_and_surroundings = cell_mask_remove_nuclei ^ dilated_inclusion_mask
    density_without_inclusions_and_surroundings = calculate_density(mitochondria_image, mask_without_inclusions_and_surroundings)

    mask_without_inclusions = cell_mask_remove_nuclei ^ inclusions_mask
    density_without_inclusions = calculate_density(mitochondria_image, mask_without_inclusions)

    if np.sum(inclusions_mask) == 0:
        density_without_inclusions = None
        density_without_inclusions_and_surroundings = None


    
    return density_cell, density_inclusions, density_surroundings, density_inclusions_and_surroundings, density_without_inclusions, density_without_inclusions_and_surroundings



def segment_nuclei_image(nuclei_image):
    """Segment the nuclei image using Cellpose."""
    masks, flows, styles, diams = model.eval(nuclei_image, diameter=None, channels=[0, 0])
    return masks




def segment_mitochondria_image(mitochondria_image):
    """Apply Otsu threshold to blurred mitochondria image."""
    blurred = gaussian(mitochondria_image, sigma=2)
    return blurred > threshold_otsu(blurred)


def calculate_mean_intensity(image, mask):
    """Compute average intensity within mask, after thresholding."""
    mask = mask * segment_mitochondria_image(image)
    masked_values = image * mask

    if np.sum(mask) == 0:
        return None

    return np.sum(masked_values) / np.sum(mask)


def calculate_density(image, mask):
    """Compute ratio of thresholded area to total mask area."""
    mask = mask.astype(bool)
    thresholded = segment_mitochondria_image(image) * mask

    if np.sum(mask) == 0:
        return None

    return np.sum(thresholded) / np.sum(mask)


In [ ]:
from matplotlib import cm
import numpy as np
from scipy.ndimage import uniform_filter

def density_box(binary_image, window_size=(10, 10), mode='constant'):
    """Local density = local mean over a uniform window."""
    img = binary_image.astype(np.float32)
    return uniform_filter(img, size=window_size, mode=mode)


def show_density_map_with_contour(green_channel, orange_channel, labeled_cells, no_inclusions, verbose=False):
    """Show density map with white cell outlines and light green inclusions."""
    orange_channel_thresholded = segment_mitochondria_image(orange_channel).astype(np.float32)
    # Compute density map and normalize to 0-1
    density_map = density_box(orange_channel_thresholded, window_size=(30,30), mode='constant')
    density_map = density_map / np.max(density_map) # dim down
   

    # Get binary mask of valid cells
    mask = labeled_cells > 0
    density_map_inside_cells = density_map * mask

    # Apply hot colormap to the density map
    
    cmap = plt.colormaps.get_cmap("hot") 
    density_rgb = cmap(density_map_inside_cells)[:, :, :3]  # Drop alpha channel

    # Draw white cell boundaries
    cell_boundaries = find_boundaries(labeled_cells, mode='thick')
    density_rgb[cell_boundaries] = [1, 1, 1]  # white

    # Generate inclusion mask and boundaries in green
    inclusion_image = generate_inclusion_image(green_channel, labeled_cells,no_inclusions).astype(bool)
    dilated_inclusion_image = binary_dilation(inclusion_image, disk(RADIUS)) * mask
    inclusion_boundaries = find_boundaries(label(dilated_inclusion_image), mode='thick')

    # Light green RGB: [0.6, 1.0, 0.6]
    density_rgb[inclusion_image] = [0.6, 1.0, 0.6]             # inclusions
    density_rgb[inclusion_boundaries] = [0.6, 1.0, 0.6]        # inclusion boundaries

    # Show the final image
    if verbose:
        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(density_map_inside_cells, cmap="hot")  # use scalar image for colorbar
        ax.imshow(density_rgb)  # show annotated RGB version
        ax.set_title("Mitochondria Density Map with Cell and Inclusion Contours")
        ax.axis("off")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Normalized Density")
        plt.show()

def show_mfi_map_with_contour(green_channel, orange_channel, labeled_cells, no_inclusions, verbose=False):
    """Show density map with white cell outlines and light green inclusions."""

    orange_channel = normalize_image(orange_channel)
    

    # Get binary mask of cells
    mask = labeled_cells > 0
    mitochondria_inside_cells = orange_channel * mask

    # Apply hot colormap 
    cmap = plt.colormaps.get_cmap("hot") 
    density_rgb = cmap(mitochondria_inside_cells)[:, :, :3]  # Drop alpha channel

    # Draw white cell boundaries
    cell_boundaries = find_boundaries(labeled_cells, mode='thick')
    density_rgb[cell_boundaries] = [1, 1, 1]  # white

    # Generate inclusion mask and boundaries in green
    inclusion_image = generate_inclusion_image(green_channel, labeled_cells, no_inclusions).astype(bool)
    dilated_inclusion_image = binary_dilation(inclusion_image, disk(RADIUS)) * mask
    inclusion_boundaries = find_boundaries(label(dilated_inclusion_image), mode='thick')

    # Light green RGB: [0.6, 1.0, 0.6]
    density_rgb[inclusion_image] = [0.6, 1.0, 0.6]             # inclusions
    density_rgb[inclusion_boundaries] = [0.6, 1.0, 0.6]        # inclusion boundaries

    # Step 6: Show the final image
    if verbose:
        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(mitochondria_inside_cells, cmap="hot")  # use scalar image for colorbar
        ax.imshow(density_rgb)  # show annotated RGB version
        ax.set_title("Mitochondria MFI Map with Cell and Inclusion Contours")
        ax.axis("off")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Normalized MFI")
        plt.show()


In [ ]:
def analyze_no_dox(image_path, basename, verbose=False):
    print(f"\n{'='*20} Analyzing {image_path} {'='*20}")
    # Load image
    czifile = CziFile(image_path)
    image = np.squeeze(czifile.read_image(S=1)[0])
    
    orange_channels = image[0]

    per_z_means = [float(np.mean(orange_channels[z])) for z in range(len(orange_channels))]
    avg_mfi_across_z = float(np.mean(per_z_means))

    # One image per row, just the mean
    return pd.DataFrame({
        "Image": [basename],
        "Orange MFI": [avg_mfi_across_z],
    })

In [ ]:
def analyze_image(image_path, basename, verbose=False):
    print(f"\n{'='*20} Analyzing {image_path} {'='*20}")


    no_inclusions = "noOA" in basename.lower()
    # Load image
    czifile = CziFile(image_path)
    image = np.squeeze(czifile.read_image(S=1)[0])
    print(f"Image shape: {image.shape}")
    if image.shape[0] < 2:
        print("Image does not have 2 channels. Skipping.")
        return None
    if image.shape[0] == 3:
        orange_channels, green_channels, blue_channels = image[0], image[1], image[2]
    if image.shape[0] == 2:
        orange_channels, green_channels = image[0], image[1]

        # make blue channel an array of zeros with same shape as green
        blue_channels = np.zeros_like(green_channels)

    if len(image.shape) < 4:
        print("Image does not have multiple z-slices. Skipping.")
        return None
    if "nodox" in basename.lower():  
        return None
        

    # Segment nuclei and mitochondria
    nuclei_masks = [segment_nuclei_image(ch) for ch in blue_channels]
    mitochondria_masks = [segment_mitochondria_image(ch) for ch in orange_channels]

    # Segment cells
    masks = segment_cells_3D(green_channels, blue_channels)
    masks = [fill_holes_and_remove_small_masks(m, min_size=1000) for m in masks]
    max_num_cells = max(np.max(m) for m in masks)

    if verbose:
        for z, mask in enumerate(masks):
            print(f"Z slice {z + 1}: {np.max(mask)} cells")
            show_images(
                [green_channels[z], mask, green_channels[z] * (mask > 0), orange_channels[z], mitochondria_masks[z]],
                titles=[f"Green {z + 1}", f"Cells {z + 1}", f"Masked {z + 1}", f"Orange {z + 1}", f"Mito Mask {z + 1}"],
                cmap=COLOR_SCHEME
            )

    # Initialize lists
    mfi_results = {
        'Avg MFI Cell': [],
        'Avg MFI Inclusions': [],
        'Avg MFI Surroundings': [],
        'Avg MFI Inclusions and Surroundings': [],
        'Avg MFI Without Inclusions': [],
        'Avg MFI Without Inclusions and Surroundings': []
    }

    density_results = {
        'Avg Density Cell': [],
        'Avg Density Inclusions': [],
        'Avg Density Surroundings': [],
        'Avg Density Inclusions and Surroundings': [],
        'Avg Density Without Inclusions': [],
        'Avg Density Without Inclusions and Surroundings': []
    }

    for z in range(len(masks)):
        print(f"\nVisualizing Z slice {z + 1}")
        show_density_map_with_contour(green_channels[z], orange_channels[z], masks[z], no_inclusions, verbose=True)
        show_mfi_map_with_contour(green_channels[z], orange_channels[z], masks[z], no_inclusions, verbose=True)
        print()

    # Iterate over cells
    for cell in range(1, max_num_cells + 1):
        if verbose:
            print(f"\nAnalyzing cell {cell}")

        # Per-z slice accumulators
        mfi_per_cell = {k: [] for k in mfi_results}
        density_per_cell = {k: [] for k in density_results}

        for z in range(len(masks)):
            cell_mask = masks[z] == cell
            if np.sum(cell_mask) == 0:
                continue

            green = green_channels[z]
            orange = orange_channels[z]
            nucleus_mask = nuclei_masks[z]
            cell_mask_wo_nuclei = cell_mask * ~nucleus_mask

            if np.sum(cell_mask_wo_nuclei) == 0:
                if verbose:
                    print(f"Z {z + 1}: Cell {cell} has only nucleus — skipped for density.")
                continue

            inclusion_mask = extract_inclusions(green, cell_mask, no_inclusions)

            # MFI
            mfi_values = calculate_mfi_around_inclusions(inclusion_mask, orange, cell_mask, verbose=verbose)
            for key, val in zip(mfi_results, mfi_values):
                if val is not None:
                    mfi_per_cell[key].append(val)

            # Density
            density_values = calculate_density_around_inclusions(inclusion_mask, orange, cell_mask_wo_nuclei, verbose=verbose)
            for key, val in zip(density_results, density_values):
                if val is not None:
                    density_per_cell[key].append(val)

        # Average over z slices for this cell
        for key in mfi_results:
            mfi_results[key].append(np.mean(mfi_per_cell[key]) if mfi_per_cell[key] else np.nan)
        for key in density_results:
            density_results[key].append(np.mean(density_per_cell[key]) if density_per_cell[key] else np.nan)

    # Compile results
    results_df = pd.DataFrame({
        'Image': [basename] * max_num_cells,
        'Cell': np.arange(1, max_num_cells + 1),
        **mfi_results,
        **density_results
    })

    return results_df


In [ ]:


def analyze_all_images(image_folder, images=None, verbose=False):
    """Analyze all .czi images in the specified folder.
    image_folder: path to the folder containing .czi images
    images: optional list of specific image filenames to analyze
    verbose: whether to print detailed logs
    """
    print("Analyzing images in folder:", image_folder)

    all_results = []
    all_results_no_dox = []
    # List all .czi files in the folder
    for filename in os.listdir(image_folder):
        if not filename.lower().endswith(".czi"):
            continue

        if images is not None and filename not in images:
            continue

        if 'zstack' not in filename.lower():
            continue

        if 'mip' in filename.lower():
            continue

        image_path = os.path.join(image_folder, filename)
        base_name = os.path.splitext(filename)[0]

       

        df = analyze_image(image_path, base_name, verbose=verbose)
      

        if df is not None and not df.empty:
            all_results.append(df)

        if "nodox" in filename.lower():
            df_no_dox = analyze_no_dox(image_path, base_name, verbose=verbose)
            if df_no_dox is not None and not df_no_dox.empty:
                all_results_no_dox.append(df_no_dox)

        if verbose:
            print(f"Completed: {filename}")
            print("-" * 60)
    
    if not all_results and not all_results_no_dox:
        print("No valid data extracted.")
        return None

    combined_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

    # Sanitize filename for output
    safe_folder_name = image_folder.replace('/', '_').replace('\\', '_').replace(' ', '_').strip('_')
    
    output_filename = f"{safe_folder_name}_analysis.xlsx"
    output_path = os.path.join(image_folder, output_filename)

    combined_df.to_excel(output_path, index=False)

    print(f"\n✅ Analysis complete. Results saved to: {output_path}")


    combined_df_no_dox = pd.concat(all_results_no_dox, ignore_index=True) if all_results_no_dox else pd.DataFrame()
    if not combined_df_no_dox.empty:
        output_filename_no_dox = f"{safe_folder_name}_no_dox_analysis.xlsx"
        output_path_no_dox = os.path.join(image_folder, output_filename_no_dox)
        combined_df_no_dox.to_excel(output_path_no_dox, index=False)
        print(f"\n✅ No-dox analysis complete. Results saved to: {output_path_no_dox}")
    

    return combined_df


In [ ]:
# folders =  ['72125_3K_dox_TMRM_hoechst/new settings','72525_1K_WT_dox_TMRM_hoechst','72825_3K_dox_TMRM_hoechst','80425_3K_dox_TMRM_hoechst','080825_3K_dox_TMRM_hoechst']
# folders = ['081225_3K_1K_WT_dox_TMRM_hoechst','081225_3K_1K_WT_dox_TMRM_hoechst/1K','081225_3K_1K_WT_dox_TMRM_hoechst/WT new settings','082025_1K_dox_TMRM_Hoeschst','082625_1k_dox_TMRM_Hoeschst','090225_1k_dox_TMRM_Hoeschst']
# folders = ['082025_1K_dox_TMRM_Hoeschst/WT','082625_1k_dox_TMRM_Hoeschst/NEW SETTINGS/1K_TMRM_hoechst','090225_1k_dox_TMRM_Hoeschst/NEW SETTINGS',
#           '090225_1k_dox_TMRM_Hoeschst/NEW SETTINGS/WT','090425_1k_WT_dox_TMRM+Hoeschst','090425_1k_WT_dox_TMRM+Hoeschst/WT','090425_1k_WT_dox_TMRM+Hoeschst',
#           '090925_1k_wt_dox_TMRM_HOESCHST','090925_1k_wt_dox_TMRM_HOESCHST/WT']

# check if all folders exist
import os
root_folder = '.'
folders = [os.path.join(root_folder, f) for f in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, f))]
for folder in folders:
    if 'Mitrotacker' not in folder:
        continue
    subfolders = [os.path.join(folder, f) for f in os.listdir(folder) if os.path.isdir(os.path.join(folder, f))]
    for subfolder in subfolders:
        print(f"Analyzing subfolder: {subfolder}")
        analyze_all_images(subfolder,verbose=VERBOSE)

